In [ ]:
#| default_exp topics

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math, re
import numpy as np
from fastcore.all import AttrDict, L, chunked, first, ifnone, patch
from fastlite import Database
from apswutils.db import Table
from litesearch.core import sql_in, rowid_sel, content_id, NP_DTYPE, upsert_all

## Schema

Three tables beside the chunk store. Entities live in a regular `get_store` so they inherit the existing ANN machinery, resolution reuses `ann_search`/`rebuild_index` rather than adding a second index path.

In [ ]:
#| export
@patch
def get_graph(self:Database,
              store:str='store',   # chunk store the graph is built over
              prefix:str=None,     # table prefix (default: '' for 'store', else '<store>_')
              ann:bool=True,       # register an ANN index on the entity store (needed for resolution)
              **kw):               # extra typed columns for the entity store
    'Create the entity/mention/edge tables for a chunk store. Idempotent; returns the tables.'
    p = prefix if prefix is not None else ('' if store == 'store' else f'{store}_')
    en, mn, eg = f'{p}entities', f'{p}mentions', f'{p}edges'
    ents = self.get_store(en, hash=True, ann=ann, kind=str, canon=str, freq=int, **kw)
    self.t[mn].create(chunk_id=str, entity_id=str, surface=str, n=int,
                      pk=('chunk_id', 'entity_id'), if_not_exists=True)
    self.t[eg].create(src=str, dst=str, rel=str, weight=float, n=int,
                      pk=('src', 'dst', 'rel'), if_not_exists=True)
    for t, c in ((mn, 'entity_id'), (mn, 'chunk_id'), (eg, 'src'), (eg, 'dst')):
        self.t[t].create_index([c], if_not_exists=True)
    return AttrDict(entities=ents, mentions=self.t[mn], edges=self.t[eg], store=self.t[store], prefix=p)

## Topics from the ANN index

usearch clusters off the HNSW graph but walks index levels, so it raises `Index too small to
cluster!` on a small corpus. `_knn_clusters` falls back to a greedy pass over the kNN graph, which
works at any size.

Labels are c-TF-IDF: term frequency inside a cluster, weighted by inverse document frequency
across clusters. Plain frequency names every cluster after the same few words.

`_knn_clusters` seeds at the densest unassigned node and claims its unassigned neighbours, so
clusters stay bounded. Label propagation collapses a dense kNN graph into one component.

In [ ]:
#| export
# connector words — cluster names built from these describe the corpus, not the cluster
_STOP = set('''the a an and or of to in is are was were be been being for on at by with from this that these those
it its as not but if then than so such can will would could should may might do does did have has had he she they
we you i him her them our your their about into over under after before between out up down off only own same too
very just also each other more most some any no nor own s t don now here there when where why how all both'''.split())
_LWORD = re.compile(r'[A-Za-z_][A-Za-z0-9_]{2,}')   # keeps identifiers whole

def ctfidf_labels(texts,        # one text per member, aligned with `lab`
                  lab,          # cluster index per member
                  k,            # number of clusters
                  top_n=4,      # terms per label
                  stop=None,    # stopword set (defaults to _STOP)
                  sep=', '):
    'Name each cluster by terms common inside it and rare across the other clusters.'
    st = _STOP if stop is None else stop
    tf = [{} for _ in range(k)]
    for t, j in zip(texts, lab):
        for w in set(_LWORD.findall((t or '').lower())):
            if w in st or len(w) < 3: continue
            tf[j][w] = tf[j].get(w, 0) + 1
    df = {}
    for d in tf:
        for w in d: df[w] = df.get(w, 0) + 1
    sz = {}
    for j in lab: sz[j] = sz.get(j, 0) + 1
    out = []
    for j, d in enumerate(tf):
        n = max(1, sz.get(j, 0))
        sc = {w: (c/n) * math.log(k/df[w]) for w, c in d.items() if df[w] < k}
        top = sorted(sc.items(), key=lambda kv: -kv[1])[:top_n]
        out.append(sep.join(w for w, _ in top))
    return out

def _usearch_clusters(idx, min_count, max_count):
    '(centroid, members) straight off the HNSW graph. usearch walks the index levels, so it needs'
    kw = {k: v for k, v in dict(min_count=min_count, max_count=max_count).items() if v}
    cl = idx.cluster(**kw)
    keys, _ = cl.centroids_popularity
    out = L()
    for ck in np.atleast_1d(keys).tolist():
        try: out.append((int(ck), [int(x) for x in np.atleast_1d(cl.members_of(ck)).tolist()]))
        except Exception: continue
    return out

def _knn_clusters(idx, keys, k=8, max_size=None, min_sim=None, dtype=np.float16):
    '(seed, members) from a greedy pass over the kNN graph harvested from HNSW. Works at any size.'
    keys = list(keys)
    if len(keys) < 4: return L()
    vecs = np.asarray(idx[np.array(keys, dtype=np.int64)], dtype=dtype).reshape(len(keys), -1)
    res = idx.search(vecs, count=min(k+1, len(keys)))
    nbr, sims = {}, []
    for i, kk in enumerate(keys):
        ks = np.atleast_1d(res.keys[i]).tolist()
        ds = np.atleast_1d(res.distances[i]).tolist()
        n = [(int(a), 1.0-float(b)) for a, b in zip(ks, ds) if int(a) != kk]
        nbr[kk] = n
        sims += [s for _, s in n]
    if min_sim is None: min_sim = float(np.median(sims)) if sims else 0.0
    cap = max_size or max(k, 4)
    dens = {kk: sum(s for _, s in v if s >= min_sim) for kk, v in nbr.items()}
    out, seen = L(), set()
    for kk in sorted(keys, key=lambda x: -dens.get(x, 0.0)):
        if kk in seen: continue
        grp = [kk]; seen.add(kk)
        for nk, s in sorted(nbr.get(kk, []), key=lambda t: -t[1]):
            if len(grp) >= cap: break
            if nk not in seen and s >= min_sim: grp.append(nk); seen.add(nk)
        out.append((int(kk), [int(x) for x in grp]))
    return out

def _cluster_groups(idx,               # usearch Index
                    keys=None,         # keys to cluster (defaults to everything in the index)
                    min_count=None,    # usearch: smallest cluster to emit
                    max_count=None,    # usearch: largest cluster to emit
                    k=8,               # neighbours per node in the kNN fallback
                    dtype=np.float16):
    '`([(centroid, members)], method)` for an index.'
    try: return _usearch_clusters(idx, min_count, max_count), 'usearch'
    except Exception:
        ks = np.atleast_1d(idx.keys).tolist() if keys is None else list(keys)
        return _knn_clusters(idx, ks, k, dtype=dtype), 'knn'

def topic_nodes(db,                # Database
                store='store',     # chunk store (must be ANN-registered)
                prefix=None,       # graph table prefix
                min_count=None,    # usearch cluster min size
                max_count=None,    # usearch cluster max size
                k=8,               # neighbours per node in the fallback kNN graph
                min_size=2,        # smallest cluster kept as a topic
                label_k=4,         # terms per topic label
                max_label_chars=4000,
                dtype=np.float16):
    'Cluster the store index into topic nodes labelled by c-TF-IDF. Returns {topics, method}.'
    g = db.get_graph(store, prefix)
    m = db._ann_meta(store)
    if not (m and m['ndim']): return dict(topics=0, method=None)
    idx = db.get_index(store)
    if idx.size < 4: return dict(topics=0, method=None)
    dt = NP_DTYPE.get(m['dtype'], dtype)
    rid2cid = {r['rowid']: r['id'] for r in db.t[store](select=f'{rowid_sel()}, id')}
    groups, method = _cluster_groups(idx, rid2cid.keys(), min_count, max_count, k, dt)
    kept = [[rid2cid[r] for r in mem if r in rid2cid] for _, mem in groups]
    kept = [c for c in kept if len(c) >= min_size]
    if not kept: return dict(topics=0, method=method)
    txt = {r['id']: r['content'] for r in db.t[store](select='id, content',
                                                      where=sql_in('id', [c for g_ in kept for c in g_[:24]]))}
    texts, lab = [], []
    for j, cids in enumerate(kept):
        for c in cids[:24]: texts.append((txt.get(c) or '')[:max_label_chars]); lab.append(j)
    names = ctfidf_labels(texts, lab, len(kept), label_k)
    ents, mens = [], []
    for j, cids in enumerate(kept):
        name = f'topic: {names[j]}'[:60] if names[j] else f'topic-{j}'
        eid = content_id(name)
        ents.append(dict(content=name, kind='topic', freq=len(cids), canon=eid))
        mens += [dict(chunk_id=c, entity_id=eid, surface=name, n=1) for c in cids]
    if ents: g.entities.insert_all(ents, upsert=True, hash_id='id', hash_id_columns=['content'])
    if mens: upsert_all(g.mentions, mens, ('chunk_id','entity_id'))
    return dict(topics=len(ents), method=method)


## Clusters and peers

`store.clusters()` maps the corpus into labelled groups. `store.peers(rowid)` returns the group one
row belongs to.

`peers` is not `ann_neighbors`. k-NN returns 15 nearest things whether or not they are related;
a cluster returns the family. `peers` falls back to `ann_neighbors` when the index cannot be
clustered, and says so in `note`.

Both return `note`, because an empty clustering and a broken index look identical to a caller.

`clusters` returns `AttrDict(clusters, method, note)`, each cluster
`AttrDict(centroid, size, label, member_keys, members)`. `method` is `usearch` or `knn`.

`peers` returns `AttrDict(hits, method, note)`.

In [ ]:
#| export
@patch
def _cluster_cached(self:Table, min_count, max_count, k, dtype):
    'Cluster once per (store, params, index size). `peers` would otherwise recluster on every call.'
    idx = self.db.get_index(self.name)
    ck = (self.name, min_count, max_count, k, idx.size)
    cache = getattr(self.db, '_cluster_cache', None)
    if cache is None: cache = self.db._cluster_cache = {}
    if ck not in cache:
        groups, method = _cluster_groups(idx, None, min_count, max_count, k, dtype)
        assign = {m: g for _, g in groups for m in g}
        cache[ck] = (groups, method, assign)
    return cache[ck]

@patch
def _member_rows(self:Table, keys, columns=None):
    'Store rows for a list of usearch keys, keyed by rowid. `content` is always fetched (labels need it).'
    if not keys: return {}
    cols = list(dict.fromkeys([c for c in (columns or []) if c != 'rowid'] + ['content']))
    sel, out = ','.join(cols + [rowid_sel()]), {}
    for b in chunked(keys, 400):
        for r in self.db.q(f'select {sel} from {self.name} where {sql_in("rowid", b)}'): out[r['rowid']] = r
    return out

@patch
def clusters(self:Table,              # ANN-registered store
             min_count:int=None,      # usearch: smallest cluster to emit
             max_count:int=None,      # usearch: largest cluster to emit
             k:int=8,                 # neighbours per node in the kNN fallback
             min_size:int=2,          # drop groups smaller than this
             label_k:int=4,           # terms per c-TF-IDF label
             members:int=24,          # member rows fetched per cluster
             columns:list=None,       # store columns to return per member row
             max_label_chars:int=4000,# per-member text budget for labelling
             dtype=np.float16):
    'The corpus grouped by embedding shape, each group named by c-TF-IDF.'
    m = self.db._ann_meta(self.name)
    if not m: return AttrDict(clusters=L(), method=None, note=f'{self.name!r} is not an ANN store')
    if not m['ndim']: return AttrDict(clusters=L(), method=None, note=f'{self.name!r} has no vectors yet')
    idx = self.db.get_index(self.name)
    if idx.size < 4: return AttrDict(clusters=L(), method=None, note=f'index holds {idx.size} vectors; too few to cluster')
    dt = NP_DTYPE.get(m['dtype'], dtype)
    groups, method, _ = self._cluster_cached(min_count, max_count, k, dt)
    kept = [(c, g) for c, g in groups if len(g) >= min_size]
    if not kept: return AttrDict(clusters=L(), method=method, note=f'no group reached min_size={min_size}')
    rows = self._member_rows([r for _, g in kept for r in g[:members]], columns)
    texts, lab = [], []
    for j, (_, g) in enumerate(kept):
        for r in g[:members]: texts.append(((rows.get(r) or {}).get('content') or '')[:max_label_chars]); lab.append(j)
    names = ctfidf_labels(texts, lab, len(kept), label_k)
    out = L(AttrDict(centroid=c, size=len(g), label=names[j] or f'group-{j}', member_keys=g,
                     members=L(rows[r] for r in g[:members] if r in rows))
            for j, (c, g) in enumerate(kept))
    return AttrDict(clusters=out.sorted(key=lambda c: -c.size), method=method,
                    note=f'{len(out)} clusters over {idx.size} vectors ({method})')

@patch
def peers(self:Table,            # ANN-registered store
          key:int,               # usearch key (rowid) whose group you want
          limit:int=25,          # members to return
          columns:list=None,     # store columns to return per member row
          min_count:int=None,    # usearch: smallest cluster to emit
          max_count:int=None,    # usearch: largest cluster to emit
          k:int=8,               # neighbours per node in the kNN fallback
          dtype=np.float16):
    'The group `key` belongs to — its family, not a ranked list of what is nearest to it.'
    nbr = lambda note: AttrDict(hits=L(self.ann_neighbors(key, limit, columns, dtype=dtype)), method='ann', note=note)
    m = self.db._ann_meta(self.name)
    if not m: return AttrDict(hits=L(), method=None, note=f'{self.name!r} is not an ANN store')
    if not m['ndim']: return AttrDict(hits=L(), method=None, note=f'{self.name!r} has no vectors yet')
    idx = self.db.get_index(self.name)
    if not idx.size or not idx.contains(key): return AttrDict(hits=L(), method=None, note=f'key {key} is not indexed')
    dt = NP_DTYPE.get(m['dtype'], dtype)
    if idx.size < 4: return nbr(f'index holds {idx.size} vectors; showing nearest neighbours')
    _, method, assign = self._cluster_cached(min_count, max_count, k, dt)
    grp = assign.get(key)
    if not grp: return nbr('row is in no cluster; showing nearest neighbours')
    mem = [x for x in grp if x != key][:limit]
    if not mem: return nbr('cluster has one member; showing nearest neighbours')
    rows, order = self._member_rows(mem, columns), {r: i for i, r in enumerate(mem)}
    return AttrDict(hits=L(rows[r] for r in sorted(rows, key=lambda r: order.get(r, 1<<30)) if r in rows),
                    method=method, note=f'cluster of {len(grp)} ({method})')


In [ ]:
# clusters(): two well-separated blobs -> two groups, each labelled by its own vocabulary
from litesearch.core import database
_cdb = database()
_cst = _cdb.get_store('cl', ann=True, ndim=8, metric='cosine')
_rs  = np.random.RandomState(0)
_a   = np.concatenate([np.ones((30,4)), np.zeros((30,4))], 1) + _rs.randn(30,8)*0.05
_b   = np.concatenate([np.zeros((30,4)), np.ones((30,4))], 1) + _rs.randn(30,8)*0.05
_txt = [f'kernel gradient tensor sample {i}' for i in range(30)] + [f'invoice ledger payment row {i}' for i in range(30)]
_cst.insert_all([{'content':t,'embedding':v.astype(np.float16).tobytes()}
                 for t,v in zip(_txt, np.concatenate([_a,_b]))])
assert _cst.rebuild_index() == 60
_cl = _cst.clusters(min_size=3)
print(_cl.note, '|', [(c.size, c.label) for c in _cl.clusters][:4])
assert len(_cl.clusters) >= 2, _cl.note
assert _cl.method in ('usearch','knn')
_labels = ' '.join(c.label for c in _cl.clusters)
assert 'tensor' in _labels or 'kernel' in _labels, _labels     # c-TF-IDF names a group after what only it says
assert all(r['rowid'] in c.member_keys for c in _cl.clusters for r in c.members)

# peers(): every member of a group has the rest of that group as its family
_c0 = first(_cl.clusters, lambda c: 'kernel' in c.label or 'tensor' in c.label)
_pr = _cst.peers(_c0.centroid, limit=5, columns=['content'])
print(_pr.note, '|', [h['content'][:24] for h in _pr.hits])
assert _pr.hits and _pr.method == _cl.method, _pr.note
assert all('kernel' in h['content'] for h in _pr.hits), _pr.hits
# a row the clustering left on its own still gets an answer -- with the fallback named in `note`
_solo = first(range(1, 61), lambda r: not _cst.peers(r).method == _cl.method)
if _solo: assert 'nearest neighbours' in _cst.peers(_solo).note

# degradation is reported, never silent
_tiny = database().get_store('tiny', ann=True, ndim=4, metric='cosine')
_tiny.insert_all([{'content':'x','embedding':np.zeros(4,dtype=np.float16).tobytes()}])
_tiny.rebuild_index()
assert _tiny.clusters().clusters == [] and 'too few' in _tiny.clusters().note
assert 'nearest neighbours' in _tiny.peers(1).note
assert database().get_store('plain').clusters().note.endswith('is not an ANN store')
assert database().get_store('empty', ann=True).clusters().note.endswith('has no vectors yet')

6 clusters over 60 vectors (knn) | [(8, 'ledger, payment, row, invoice'), (8, 'tensor, sample, gradient, kernel'), (7, 'tensor, sample, gradient, kernel'), (5, 'ledger, payment, row, invoice')]
cluster of 8 (knn) | ['kernel gradient tensor s', 'kernel gradient tensor s', 'kernel gradient tensor s', 'kernel gradient tensor s', 'kernel gradient tensor s']


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()